# 08 — Comparación controlada: recorte RGB vs. mapa de 21 puntos

**Pregunta:** para la clasificación estática de ASL, ¿MobileNetV2 funciona mejor con un recorte RGB de la mano detectada por MediaPipe o con un mapa RGB de sus 21 landmarks?

## Diseño del experimento

- Se usa una muestra balanceada de **200 imágenes por clase**: 24 letras estáticas (`A–I`, `K–Y`) y `nothing` (sin mano), 5.000 imágenes como máximo.
- Se excluyen `J` y `Z` porque requieren movimiento; también `space` y `del` porque son controles del dataset, no letras estáticas.
- MediaPipe procesa cada imagen una sola vez y genera **las dos representaciones**: recorte y mapa. Ambos modelos reciben exactamente las mismas imágenes, etiquetas y particiones 70/15/15.
- Se usa MobileNetV2 para ambos modelos. Esto compara la representación de entrada manteniendo fija la arquitectura. El preentrenamiento ImageNet puede favorecer al RGB natural más que al mapa sintético; se debe reportar esa limitación.
- Los fallos de detección se excluyen de entrenamiento/validación para letras; en la evaluación end-to-end se cuentan como error (se asigna `nothing`).
- Las fotos de `data/mis_fotos` se mantienen como evaluación externa opcional y no se usan para entrenar.

El resultado es un **experimento rápido de selección**, no una evaluación final del dataset combinado. Las particiones aleatorias de Kaggle pueden contener condiciones visuales parecidas; la prueba externa ayuda a observar transferencia, pero no sustituye una prueba por participantes independientes.

In [ ]:
# Google Colab. Si MediaPipe ya está instalado desde el notebook 06, la instalación se omite.
%pip install -q mediapipe==0.10.35 scikit-learn

from pathlib import Path
from IPython.display import display
import os

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if not IN_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
import csv
import json
import random
import urllib.request
import warnings

import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
MAX_PER_CLASS = 200
IMAGE_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 6
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

if IN_KAGGLE:
    INPUT_ROOT = Path("/kaggle/input")  # Entradas de Kaggle son de solo lectura.
    OUTPUT_DIR = Path("/kaggle/working/comparacion_representacion_200")
    candidates = [INPUT_ROOT, *INPUT_ROOT.glob("*"), *INPUT_ROOT.glob("*/*"), *INPUT_ROOT.glob("*/*/*")]
    RAW_DIR = next((p for p in candidates if p.is_dir() and all((p / c).is_dir() for c in ("A", "B", "nothing"))), None)
    if RAW_DIR is None:
        raise FileNotFoundError(
            "No encontré las carpetas A, B y nothing bajo /kaggle/input. "
            f"Entradas conectadas: {[p.name for p in INPUT_ROOT.iterdir()]}. "
            "Adjunta el dataset ASL Alphabet con Add Input."
        )
    own_candidates = [p for p in INPUT_ROOT.rglob("mis_fotos") if p.is_dir()]
    OWN_PHOTOS_DIR = own_candidates[0] if own_candidates else None
    MODEL_ASSET = OUTPUT_DIR / "hand_landmarker.task"
else:
    DRIVE_ROOT = Path("/content/drive/MyDrive/proyecto-ia2")
    RAW_DIR = DRIVE_ROOT / "data/raw/asl_alphabet_train/asl_alphabet_train"
    OWN_PHOTOS_DIR = DRIVE_ROOT / "data/mis_fotos"
    OUTPUT_DIR = DRIVE_ROOT / "data/processed/comparacion_representacion_200"
    MODEL_ASSET = DRIVE_ROOT / "data/processed/landmark_map_experiment/hand_landmarker.task"

STATIC_CLASSES = [chr(code) for code in range(ord("A"), ord("Z") + 1) if chr(code) not in {"J", "Z"}]
CLASSES = STATIC_CLASSES + ["nothing"]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASSES)}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"No encuentro el dataset en: {RAW_DIR}")
if not MODEL_ASSET.exists():
    MODEL_ASSET.parent.mkdir(parents=True, exist_ok=True)
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_ASSET)

print("Entorno:", "Kaggle" if IN_KAGGLE else "Colab")
print("RAW_DIR:", RAW_DIR)
print("OWN_PHOTOS_DIR:", OWN_PHOTOS_DIR)
print(f"Clases ({len(CLASSES)}): {CLASSES}")
print("Máximo de imágenes:", MAX_PER_CLASS * len(CLASSES))
print("Salida:", OUTPUT_DIR)

## 1. Definir el procesamiento

Esta celda contiene las funciones compartidas de selección, MediaPipe, recorte y mapa. Ejecútala una vez.

In [ ]:
FINGER_COLORS = [(255, 90, 90), (255, 210, 40), (80, 235, 100), (50, 190, 255), (220, 90, 255)]
FINGER_EDGES = [[(0,1),(1,2),(2,3),(3,4)], [(0,5),(5,6),(6,7),(7,8)], [(0,9),(9,10),(10,11),(11,12)], [(0,13),(13,14),(14,15),(15,16)], [(0,17),(17,18),(18,19),(19,20)]]
PALM_EDGES = [(5,9),(9,13),(13,17)]
POINT_COLORS = {**{i:FINGER_COLORS[0] for i in range(1,5)}, **{i:FINGER_COLORS[1] for i in range(5,9)}, **{i:FINGER_COLORS[2] for i in range(9,13)}, **{i:FINGER_COLORS[3] for i in range(13,17)}, **{i:FINGER_COLORS[4] for i in range(17,21)}, 0:(255,255,255)}


def find_class_dir(root, label):
    for folder in Path(root).iterdir():
        if folder.is_dir() and folder.name.strip().lower() == label.lower():
            return folder
    return None


def select_records(root, max_per_class=MAX_PER_CLASS):
    root = Path(root)
    records = []
    for label in CLASSES:
        folder = find_class_dir(root, label)
        if folder is None:
            if label == "nothing" and root == RAW_DIR:
                raise FileNotFoundError(f"Falta la carpeta de clase `nothing` en {root}")
            continue
        files = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        if max_per_class is not None and len(files) > max_per_class:
            rng = np.random.default_rng(SEED + sum(map(ord, label)))
            ids = np.sort(rng.choice(len(files), size=max_per_class, replace=False))
            files = [files[i] for i in ids]
        records.extend({"source": p, "source_rel": p.relative_to(root).as_posix(), "label": label} for p in files)
    if not records:
        raise FileNotFoundError(f"No encontré imágenes de las clases esperadas dentro de {root}")
    counts = pd.Series([r["label"] for r in records]).value_counts().reindex(CLASSES, fill_value=0)
    display(counts.rename("imágenes seleccionadas").to_frame())
    return records


def landmarks_to_map(hand_landmarks):
    xy = np.asarray([(p.x, p.y) for p in hand_landmarks], dtype=np.float32)
    if xy.shape != (21, 2) or not np.isfinite(xy).all():
        return None
    xy -= xy[0]
    palm_scale = float(np.linalg.norm(xy[9]))
    if palm_scale < 1e-6:
        return None
    xy /= palm_scale
    extent = float(np.max(np.abs(xy)))
    if extent < 1e-6:
        return None
    xy = xy / extent * 0.40 * (IMAGE_SIZE - 1) + (IMAGE_SIZE - 1) / 2
    points = [(int(round(x)), int(round(y))) for x, y in xy]
    canvas = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    width, radius = max(2, IMAGE_SIZE // 70), max(3, IMAGE_SIZE // 48)
    for edges, color in zip(FINGER_EDGES, FINGER_COLORS):
        for a, b in edges:
            draw.line((points[a], points[b]), fill=color, width=width)
    for a, b in PALM_EDGES:
        draw.line((points[a], points[b]), fill=(235, 235, 235), width=width)
    for i, (x, y) in enumerate(points):
        draw.ellipse((x-radius, y-radius, x+radius, y+radius), fill=POINT_COLORS[i], outline=(255,255,255))
    return canvas


def crop_from_landmarks(image, hand_landmarks, margin=0.25):
    w, h = image.size
    xs = np.asarray([p.x * w for p in hand_landmarks], dtype=np.float32)
    ys = np.asarray([p.y * h for p in hand_landmarks], dtype=np.float32)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    cx, cy = (x0+x1)/2, (y0+y1)/2
    side = max(x1-x0, y1-y0) * (1 + 2*margin)
    side = max(float(side), 2.0)
    left, top = int(np.floor(cx-side/2)), int(np.floor(cy-side/2))
    right, bottom = left + int(np.ceil(side)), top + int(np.ceil(side))
    pad = (max(0,-left), max(0,-top), max(0,right-w), max(0,bottom-h))
    if any(pad):
        image = ImageOps.expand(image, border=pad, fill=(0,0,0))
        left += pad[0]; right += pad[0]; top += pad[1]; bottom += pad[1]
    return image.crop((left, top, right, bottom)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)


def build_detector():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


def prepare_representation_set(input_root, subset_name, max_per_class=MAX_PER_CLASS, progress_every=250):
    records = select_records(input_root, max_per_class=max_per_class)
    base = OUTPUT_DIR / subset_name
    crops_root, maps_root = base / "crops", base / "maps"
    manifest_path = base / "manifest.csv"
    crops_root.mkdir(parents=True, exist_ok=True)
    maps_root.mkdir(parents=True, exist_ok=True)
    existing = {}
    if manifest_path.exists():
        old = pd.read_csv(manifest_path).drop_duplicates("source_rel", keep="last")
        existing = {str(row.source_rel): row._asdict() for row in old.itertuples(index=False)}
    write_header = not manifest_path.exists() or manifest_path.stat().st_size == 0
    detector = build_detector()
    try:
        with manifest_path.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=["source_rel","label","crop_path","map_path","detected","error"])
            if write_header:
                writer.writeheader()
            for i, record in enumerate(records, start=1):
                label, rel, source = record["label"], record["source_rel"], record["source"]
                stem = Path(rel).stem
                crop_rel, map_rel = f"{label}/{stem}_crop.png", f"{label}/{stem}_map.png"
                crop_path, map_path = crops_root / crop_rel, maps_root / map_rel
                old = existing.get(rel)
                if old and crop_path.is_file() and map_path.is_file():
                    continue
                crop_path.parent.mkdir(parents=True, exist_ok=True)
                map_path.parent.mkdir(parents=True, exist_ok=True)
                error = ""
                try:
                    with Image.open(source) as opened:
                        image = ImageOps.exif_transpose(opened).convert("RGB")
                    if label == "nothing":
                        detected, crop, mapped = False, Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE)), Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE))
                    else:
                        rgb = np.asarray(image)
                        result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
                        if result.hand_landmarks:
                            hand = result.hand_landmarks[0]
                            mapped = landmarks_to_map(hand)
                            detected = mapped is not None
                            crop = crop_from_landmarks(image, hand) if detected else Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE))
                            if not detected:
                                mapped = Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE))
                        else:
                            detected = False
                            crop = Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE))
                            mapped = Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE))
                    crop.save(crop_path, format="PNG")
                    mapped.save(map_path, format="PNG")
                except Exception as exc:
                    detected = False
                    error = f"{type(exc).__name__}: {exc}"
                    Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE)).save(crop_path)
                    Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE)).save(map_path)
                writer.writerow({"source_rel":rel,"label":label,"crop_path":crop_rel,"map_path":map_rel,"detected":int(detected),"error":error})
                if i % 100 == 0:
                    stream.flush()
                if i % progress_every == 0 or i == len(records):
                    stream.flush()
                    print(f"Procesadas {i:,}/{len(records):,} | {rel}")
    finally:
        detector.close()
    return pd.read_csv(manifest_path).drop_duplicates("source_rel", keep="last")


## 2. Generar el piloto

Ejecuta esta celda: procesa hasta cinco imágenes por clase. La celda siguiente muestra los dos formatos para inspección visual.

In [ ]:
pilot_df = prepare_representation_set(RAW_DIR, "pilot", max_per_class=5)
print(f"Piloto listo: {len(pilot_df)} imágenes. Revisa recortes y mapas abajo.")

In [ ]:
import matplotlib.pyplot as plt

shown = pilot_df[pilot_df.detected.astype(bool)].sample(
    min(8, int(pilot_df.detected.sum())), random_state=SEED
)
if shown.empty:
    print("MediaPipe no detectó manos en el piloto. Revisa RAW_DIR y las imágenes antes de continuar.")
else:
    fig, axes = plt.subplots(len(shown), 2, figsize=(8, max(3, 2.5 * len(shown))))
    if len(shown) == 1:
        axes = np.asarray([axes])
    for ax_row, row in zip(axes, shown.itertuples(index=False)):
        ax_row[0].imshow(Image.open(OUTPUT_DIR / "pilot" / "crops" / row.crop_path))
        ax_row[0].set_title(f"Recorte | {row.label}"); ax_row[0].axis("off")
        ax_row[1].imshow(Image.open(OUTPUT_DIR / "pilot" / "maps" / row.map_path))
        ax_row[1].set_title(f"Mapa | {row.label}"); ax_row[1].axis("off")
    plt.tight_layout(); plt.show()

## 3. Procesar 200 imágenes por clase

Después de revisar el piloto, ejecuta esta celda. Genera hasta 5.000 imágenes, reanuda desde el manifiesto si se interrumpe y guarda recortes/mapas en Drive o `/kaggle/working`.

In [ ]:
train_df = prepare_representation_set(
    RAW_DIR,
    "train_200",
    max_per_class=MAX_PER_CLASS,
    progress_every=250,
)
print(f"Muestra lista: {len(train_df):,} imágenes | salida: {OUTPUT_DIR / 'train_200'}")

## 4. Crear una partición común

Ejecuta esta celda después del procesamiento. Los dos modelos usarán exactamente estas particiones estratificadas 70/15/15.

In [ ]:
split_path = OUTPUT_DIR / "splits_200.csv"
all_paths = set(train_df.source_rel.astype(str))
splits = pd.read_csv(split_path) if split_path.exists() else None
if splits is None or set(splits.source_rel.astype(str)) != all_paths:
    train_val, test = train_test_split(
        train_df, test_size=0.15, stratify=train_df.label, random_state=SEED
    )
    train, val = train_test_split(
        train_val, test_size=0.15 / 0.85, stratify=train_val.label, random_state=SEED
    )
    splits = pd.concat([
        train.assign(split="train"), val.assign(split="val"), test.assign(split="test")
    ], ignore_index=True)
    splits.to_csv(split_path, index=False)

display(pd.crosstab(splits.split, splits.label))
print("Tamaños:", splits.split.value_counts().to_dict())
sign_train = splits[(splits.split == "train") & (splits.label != "nothing") & splits.detected.astype(bool)]
nohand_train = splits[(splits.split == "train") & (splits.label == "nothing")]
sign_val = splits[(splits.split == "val") & (splits.label != "nothing") & splits.detected.astype(bool)]
nohand_val = splits[(splits.split == "val") & (splits.label == "nothing")]
train_rows = pd.concat([sign_train, nohand_train]).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_rows = pd.concat([sign_val, nohand_val]).reset_index(drop=True)
test_rows = splits[splits.split == "test"].reset_index(drop=True)
print(f"Filas train/val usables: {len(train_rows):,}/{len(val_rows):,}; test end-to-end: {len(test_rows):,}")

## 5. Definir el entrenamiento y la evaluación

Ejecuta una vez para cargar las funciones. Los modelos son MobileNetV2 idénticos salvo por la representación de entrada.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_dataset(frame, modality, training=False, subset_name="train_200"):
    path_col = "crop_path" if modality == "crop" else "map_path"
    root = OUTPUT_DIR / subset_name / ("crops" if modality == "crop" else "maps")
    paths = [str(root / p) for p in frame[path_col].astype(str)]
    labels = frame.label.map(CLASS_TO_ID).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def load(path, label):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE), antialias=True)
        return tf.cast(img, tf.float32), label

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if training:
        is_map = modality == "map"
        def augment(img, label):
            foreground = tf.reduce_max(img, axis=-1, keepdims=True) > 0.5
            out = tf.image.random_contrast(img, 0.85, 1.15)
            out = tf.image.random_brightness(out, max_delta=15.0)
            out = tf.clip_by_value(out, 0.0, 255.0)
            if is_map:
                out = tf.where(foreground, out, tf.zeros_like(out))
            return out, label
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


def build_mobilenet():
    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name="input_image")
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    backbone = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax", dtype="float32")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model, backbone


def fit_representation(modality):
    run_dir = OUTPUT_DIR / "experiments" / f"mobilenetv2_{modality}"
    run_dir.mkdir(parents=True, exist_ok=True)
    train_ds = make_dataset(train_rows, modality, training=True)
    val_ds = make_dataset(val_rows, modality)
    model, backbone = build_mobilenet()

    def callbacks_for(phase):
        return [
            tf.keras.callbacks.BackupAndRestore(backup_dir=str(run_dir / f"backup_{phase}")),
            tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=2, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
            tf.keras.callbacks.ModelCheckpoint(
                str(run_dir / f"best_{phase}.keras"), monitor="val_accuracy",
                mode="max", save_best_only=True,
            ),
        ]

    head_hist = model.fit(
        train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
        callbacks=callbacks_for("head"), verbose=2,
    )
    head_score = max(head_hist.history.get("val_accuracy", [0.0]))
    head_path = run_dir / "head_model.keras"
    model.save(head_path)

    backbone.trainable = True
    for layer in backbone.layers[:-30]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    fine_hist = model.fit(
        train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
        callbacks=callbacks_for("finetune"), verbose=2,
    )
    fine_score = max(fine_hist.history.get("val_accuracy", [0.0]))
    if fine_score < head_score:
        model = tf.keras.models.load_model(head_path)
    model.save(run_dir / "model.keras")
    with open(run_dir / "classes.json", "w", encoding="utf-8") as f:
        json.dump(CLASSES, f, ensure_ascii=False, indent=2)
    history = pd.concat([
        pd.DataFrame(head_hist.history).assign(phase="head"),
        pd.DataFrame(fine_hist.history).assign(phase="finetune"),
    ], ignore_index=True)
    history.to_csv(run_dir / "history.csv", index=False)
    return model, run_dir


def evaluate_end_to_end(model, frame, modality, output_prefix, subset_name="train_200"):
    ds = make_dataset(frame, modality, subset_name=subset_name)
    probabilities = model.predict(ds, verbose=0)
    y_true = frame.label.map(CLASS_TO_ID).to_numpy()
    y_pred = probabilities.argmax(axis=1)
    detected = frame.detected.astype(bool).to_numpy()
    y_pred[~detected] = CLASS_TO_ID["nothing"]
    sign_mask = frame.label.to_numpy() != "nothing"
    metrics = {
        "modality": modality,
        "n_images": int(len(frame)),
        "accuracy_end_to_end": float(accuracy_score(y_true, y_pred)),
        "macro_f1_end_to_end": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "hand_detection_rate_on_letters": float(detected[sign_mask].mean()) if sign_mask.any() else 0.0,
        "classification_report": classification_report(
            y_true, y_pred, labels=list(range(len(CLASSES))), target_names=CLASSES,
            output_dict=True, zero_division=0,
        ),
    }
    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    with open(str(output_prefix) + "_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    pd.DataFrame({
        "true": [CLASSES[i] for i in y_true],
        "prediction": [CLASSES[i] for i in y_pred],
    }).to_csv(str(output_prefix) + "_predictions.csv", index=False)
    return metrics


## 6. Entrenar los dos modelos

Ejecuta la celda siguiente para entrenar primero MobileNetV2 con recortes y luego con mapas. Esta es la parte que tarda más. Los checkpoints por época y los modelos se guardan en Drive (Colab) o `/kaggle/working` (Kaggle).

In [ ]:
models, metrics = {}, []
for modality in ["crop", "map"]:
    print(f"\n=== Entrenando MobileNetV2 con {modality} ===")
    models[modality], run_dir = fit_representation(modality)
    result = evaluate_end_to_end(
        models[modality], test_rows, modality, run_dir / "internal_test"
    )
    metrics.append(result)
    print(f"{modality}: accuracy={result['accuracy_end_to_end']:.4f} | "
          f"macro F1={result['macro_f1_end_to_end']:.4f} | "
          f"detección={result['hand_detection_rate_on_letters']:.1%}")

comparison = pd.DataFrame([{k: v for k, v in m.items() if k != "classification_report"} for m in metrics])
comparison.to_csv(OUTPUT_DIR / "comparison_metrics.csv", index=False)
display(comparison)

## 7. Evaluación externa opcional

Ejecuta la celda de abajo después del entrenamiento. Si `data/mis_fotos` (o un input Kaggle con carpeta `mis_fotos`) no existe, la celda informa que se omite. Las fotos propias se usan solo para evaluación.

In [ ]:
if OWN_PHOTOS_DIR is None or not Path(OWN_PHOTOS_DIR).is_dir():
    print("No se encontró la carpeta opcional de fotos propias; se omite esta evaluación.")
else:
    own_df = prepare_representation_set(OWN_PHOTOS_DIR, "own_photos", max_per_class=None)
    own_results = []
    for modality in ["crop", "map"]:
        model = tf.keras.models.load_model(
            OUTPUT_DIR / "experiments" / f"mobilenetv2_{modality}" / "model.keras"
        )
        own_results.append(evaluate_end_to_end(
            model, own_df, modality,
            OUTPUT_DIR / "experiments" / f"mobilenetv2_{modality}" / "own_photos",
            subset_name="own_photos",
        ))
    own_comparison = pd.DataFrame([
        {k: v for k, v in result.items() if k != "classification_report"}
        for result in own_results
    ])
    own_comparison.to_csv(OUTPUT_DIR / "own_photos_comparison.csv", index=False)
    display(own_comparison)

## Cómo decidir

Compara macro F1 y accuracy end-to-end, tasa de detección y errores por clase. La comparación aísla la representación de entrada usando la misma arquitectura y particiones. El preentrenamiento ImageNet puede favorecer al recorte RGB sobre el mapa sintético; por eso interpreta el resultado como “qué entrada funciona mejor con MobileNetV2 en esta configuración”. Para la evaluación final, conserva fotos/participantes no usados en entrenamiento y mide latencia si el objetivo es tiempo real.